This notebook is for visualization

In [104]:
import pandas as pd
import matplotlib.pyplot as plt
import folium
from folium.plugins import MarkerCluster

We take in all the gold serving layers to be part of the analysis

In [105]:
RAW_DIRECTORY =  r"A:\JnJ_assignment_data_science\Data\Gold"
BRONZE_DIRECTORY = r"A:\JnJ_assignment_data_science\Data\Bronze"
employees_df = pd.read_csv(RAW_DIRECTORY + r"\Gold_Employees.csv")
area_df = pd.read_csv(RAW_DIRECTORY + r"\Gold_Area_Summary.csv")
raw_employees_df = pd.read_csv(BRONZE_DIRECTORY + r"\Employees_table.csv")
possible_stations_df = pd.read_csv(BRONZE_DIRECTORY + r"\Possible_generated_stations.csv")


To show the Summary of the count of the employees under the timeline with respect to the percentage


In [106]:
summary_commute_group = pd.DataFrame({
    "Time taken": [
        "Within 30 minutes",
        "Within 45 minutes",
        "Within 60 minutes",
        "Over 60 minutes"
    ],
    "Employees": [
        (employees_df["total_time"] <= 30).sum(),
        (employees_df["total_time"] <= 45).sum(),
        (employees_df["total_time"] <= 60).sum(),
        (employees_df["total_time"] > 60).sum()
    ]
})

summary_commute_group["Percentage"] = (summary_commute_group["Employees"] / (len(employees_df)) * 100).round(2)

display(summary_commute_group)

,Time taken,Employees,Percentage
0,Within 30 minutes,18,3.61
1,Within 45 minutes,136,27.31
2,Within 60 minutes,407,81.73
3,Over 60 minutes,91,18.27


To show the summary of employees with high chances for deutschland ticket adoption

In [107]:
summary_Deutschlandticket_summary = (employees_df["Deutschlandticket_Metric"].value_counts().reset_index())
summary_Deutschlandticket_summary.columns = ["Deutschlandticket_Metric", "Employees"]
summary_Deutschlandticket_summary["Percentage"] = (summary_Deutschlandticket_summary["Employees"] / (len(employees_df)) * 100).round(2)
display(summary_Deutschlandticket_summary)

,Deutschlandticket_Metric,Employees,Percentage
0,Very High,357,71.69
1,High,78,15.66
2,Low,52,10.44
3,Very Low,11,2.21


This shows the count for connectivity areas 

In [108]:
summary_connectivity = (area_df["Connectivity_category"].value_counts().reset_index())

summary_connectivity.columns = ["Connectivity_category", "Number_of_Areas"]

display(summary_connectivity)

,Connectivity_category,Number_of_Areas
0,Less attractive public transport,17
1,Strong public transport connectivity,13


The locations with strongest connectivity

In [109]:
strong_areas = area_df.sort_values(by="Connectivity_score",ascending=False).head(10)
display(strong_areas)

,Location,Employees,Avg_commute_time,Avg_distance_from_home_to_station,High_adoption_percentage,Unique_stations,Main_station,Connectivity_score,Connectivity_category
4,Halstenbek,18,49.09,1.61,100.00,4,Halstenbek,3.3,Strong public transport connectivity
9,Hamburg-Nord,18,49.98,1.28,100.00,7,Winterhude / Hudtwalckerstraße,3.3,Strong public transport connectivity
16,Niendorf,18,45.42,1.45,100.00,5,Niendorf Markt,3.3,Strong public transport connectivity
29,Winterhude,18,48.59,1.24,100.00,12,Winterhude / Hudtwalckerstraße,3.3,Strong public transport connectivity
14,Lokstedt,18,46.92,1.41,100.00,5,Stellingen,3.3,Strong public transport connectivity
11,Henstedt-Ulzburg,12,48.82,1.91,100.00,3,Henstedt-Ulzburg,3.2,Strong public transport connectivity
21,Poppenbüttel,12,44.66,1.40,100.00,3,Poppenbüttel,3.2,Strong public transport connectivity
19,Norderstedt-Harksheide,18,35.09,2.26,66.67,5,Norderstedt Harksheide,3.2,Strong public transport connectivity
18,Norderstedt-Garstedt,18,30.56,1.66,33.33,6,Norderstedt Harksheide,3.1,Strong public transport connectivity
17,Norderstedt,30,31.35,1.83,33.33,9,Norderstedt Harksheide,3.1,Strong public transport connectivity


The locations with weak connectivity

In [110]:
weak_areas = area_df.sort_values(by="Connectivity_score",ascending=True).head(10)
display(weak_areas)

,Location,Employees,Avg_commute_time,Avg_distance_from_home_to_station,High_adoption_percentage,Unique_stations,Main_station,Connectivity_score,Connectivity_category
15,Lüneburg,12,128.11,2.54,0.00,1,Lüneburg,1.3,Less attractive public transport
3,Elmshorn,12,69.88,3.14,100.00,2,Elmshorn,1.7,Less attractive public transport
2,Bergedorf,12,76.02,3.06,91.67,2,Bergedorf,1.7,Less attractive public transport
28,Wedel,12,68.62,3.32,100.00,2,Wedel,1.7,Less attractive public transport
8,Hamburg-Harburg,12,75.03,2.16,100.00,1,Harburg,1.9,Less attractive public transport
27,Uetersen,12,66.41,2.59,100.00,2,Uetersen,2.0,Less attractive public transport
1,Ammersbek,12,52.40,2.31,100.00,2,Ammersbek,2.4,Less attractive public transport
12,Kaltenkirchen,12,57.43,2.46,100.00,2,Kaltenkirchen,2.4,Less attractive public transport
20,Pinneberg,18,50.86,2.18,100.00,2,Pinneberg,2.4,Less attractive public transport
26,Tornesch,12,59.10,2.06,100.00,3,Tornesch,2.5,Less attractive public transport


Part B : Map visualization

We perform a *left Join* between the gold employees df -> employees_df in order to extract the lat and long, this helps with the identification of the spots

In [111]:
dim_map = employees_df.merge(raw_employees_df[["EMP_ID", "Lat", "Long"]],on="EMP_ID",how="left")

The target location which is the office location is mapped using folium and marker is used in order to make sure the pointer is visible in the map

In [112]:
target_lat = 53.6907
target_long = 9.9806
mapping = folium.Map(location=[target_lat, target_long],zoom_start=10,tiles="OpenStreetMap")
folium.Marker(location=[target_lat, target_long],popup="Johnson & Johnson Medical GmbH",tooltip="Workplace",
    icon=folium.Icon(color="red", icon="briefcase", prefix="fa")).add_to(mapping)

The stations are grouped into a single cluster category

In [113]:
grouped_station = MarkerCluster(name="Stations").add_to(mapping)
for idx, row in possible_stations_df.iterrows():
    folium.Marker(location=[row["lat"], row["long"]],popup=row["station"],tooltip=row["station"],
        icon=folium.Icon(color="blue", icon="train", prefix="fa")).add_to(grouped_station)

So based on the **Deutschland_metric**, I have given the color code which could eb shown for high possibiity for deustchland ticket procurement 

In [114]:
def adoption_color(metric):
    if metric == "Very High":
        return "green"
    elif metric == "High":
        return "orange"
    elif metric == "Low":
        return "gray"
    else:
        return "lightgray"

In [115]:
grouped_employee = MarkerCluster(name="Employees").add_to(mapping)
for _, row in dim_map.iterrows():
    popup_text = f"""
    <b>EMP_ID:</b> {row['EMP_ID']}<br>
    <b>Location:</b> {row['Location']}<br>
    <b>Possible_nearest_station:</b> {row['Possible_nearest_station']}<br>
    <b>total_time:</b> {row['total_time']} minutes<br>
    <b>Commute_group:</b> {row['commute_group']}<br>
    <b>Deutschlandticket_Metric:</b> {row['Deutschlandticket_Metric']}
    """

    folium.CircleMarker(location=[row["Lat"], row["Long"]],radius=5, popup=folium.Popup(popup_text, max_width=300),
         color=adoption_color(row["Deutschlandticket_Metric"]),fill=True,fill_opacity=0.7).add_to(grouped_employee)


In [116]:
mapping